# Matrix Chain Multiplication (Dynamic Programming)

## Introduction
Matrix multiplication is **associative**, so for a chain such as `A × B × C × D` the final result is the same no matter how we place the parentheses. The **cost** of computing it is not the same, though.

Multiplying a `p × q` matrix by a `q × r` matrix takes about `p · q · r` scalar multiplications. Different parenthesizations create different intermediate matrices, so the total cost can differ a lot.

## Problem Statement
Given a chain of matrices `M₀, M₁, …, Mₙ₋₁`, where matrix `Mᵢ` has dimensions `pᵢ × pᵢ₊₁`, find the **minimum number of scalar multiplications** needed to compute the full product.

> We are **not** multiplying the matrices here. We only decide the cheapest order of multiplication, so only the *shapes* matter, not the values.

## Approach
We use **bottom-up dynamic programming**:
1. Define `dp[i][j]` as the minimum cost of multiplying the sub-chain `Mᵢ … Mⱼ`.
2. Try every split point `k` between `i` and `j`.
3. Build answers for short chains first, then use them for longer chains.

## Notebook Outline
1. Import libraries
2. Define the `Min_multi` function
3. Create the input matrices
4. Run the function
5. Results and analysis

## 1. Importing Libraries
We import **NumPy** (`np`). It is used for two things:
- `np.random.rand` to create sample matrices with the shapes we want.
- `np.zeros` and `np.inf` to build and initialize the DP table.

In [1]:
import numpy as np

## 2. The `Min_multi` Function

This function takes a list of NumPy matrices and returns the **minimum number of scalar multiplications** needed to multiply them all together.

### How the code works

**Step 1: Extract the shapes**
```python
shape = [x.shape for x in mtrxs]
```
Each matrix is reduced to its `(rows, cols)` tuple. `shape[i][0]` is the number of rows of matrix `i`, and `shape[i][1]` is its number of columns.

**Step 2: Create the DP table**
```python
dp = np.zeros((n, n))
```
`dp[i][j]` holds the minimum cost of multiplying matrices `i` through `j`. A single matrix needs no multiplication, so the diagonal `dp[i][i]` stays `0`.

**Step 3: Loop over chain lengths**
```python
for l in range(1, n):        # l = gap between i and j
    for i in range(n - l):   # starting matrix
        j = i + l            # ending matrix
```
We solve all chains of length 2 first (`l = 1`), then length 3, and so on, up to the full chain. This guarantees the smaller sub-problems are already solved when needed.

**Step 4: Try every split point**
```python
dp[i][j] = np.inf
for k in range(i, j):
    val = dp[i][k] + dp[k+1][j] + shape[i][0] * shape[k][1] * shape[j][1]
    dp[i][j] = min(dp[i][j], val)
```
Splitting the chain `Mᵢ … Mⱼ` at `k` gives two groups, `(Mᵢ … Mₖ)` and `(Mₖ₊₁ … Mⱼ)`. The cost has three parts:

| Term | Meaning |
|---|---|
| `dp[i][k]` | Cost of computing the left group |
| `dp[k+1][j]` | Cost of computing the right group |
| `shape[i][0] * shape[k][1] * shape[j][1]` | Cost of multiplying the two results together |

The last term works because the left product has `shape[i][0]` rows and `shape[k][1]` columns, and the right product has `shape[k][1]` rows and `shape[j][1]` columns.

**Step 5: Return the answer**
```python
return dp[0][n-1]
```
The top-right cell of the table covers the entire chain.

In [2]:
def Min_multi(mtrxs):
    shape = []
    for x in mtrxs :
        shape.append(x.shape)
    n = len(mtrxs)
    dp = np.zeros((n,n))
    s =  np.zeros((n,n),dtype = int)
    for l in range(1,n):
        for i in range(n-l):
            j = i+l
            dp[i][j] = np.inf
            for k in range(i,j):
                val = dp[i][k] + dp[k+1][j] + shape[i][0] * shape[k][1] * shape[j][1]
                if(val < dp[i][j]):
                    dp[i][j] = val
                    s[i][j] = k
                    

    print("------------------------------------------------------------------------------")
    print(s)
    print("------------------------------------------------------------------------------")
    
    return dp[0][n-1]



## 3. Creating the Input Matrices

We create four random matrices to test the function:

| Matrix | Shape | Rows × Cols |
|---|---|---|
| `A` | 10 × 20 | 10 × 20 |
| `B` | 20 × 9 | 20 × 9 |
| `C` | 9 × 15 | 9 × 15 |
| `D` | 15 × 8 | 15 × 8 |

### Why these shapes are valid
For a chain to be multiplicable, the **columns of each matrix must equal the rows of the next one**:

`A(10×20) · B(20×9) · C(9×15) · D(15×8)`

The inner dimensions (`20`, `9`, `15`) match, so the chain is valid and the result would be a `10 × 8` matrix.

### Dimension array
In the classic textbook formulation, this chain is written as the dimension array:

`p = [10, 20, 9, 15, 8]`

> The values inside the matrices are random and irrelevant to the answer. The algorithm only uses the **shapes**.

The matrices are stored together in the list `matrixes = [A, B, C, D]`, which is what gets passed to `Min_multi`.

In [3]:
A = np.random.rand(10, 20)
B = np.random.rand(20, 9)
C = np.random.rand(9, 15)
D = np.random.rand(15, 8)

matrixes = [A, B, C, D]


## 4. Running the Algorithm

We call `Min_multi(matrixes)` and print the result. The printed number is the **minimum total count of scalar multiplications** over all possible ways to parenthesize `A · B · C · D`.

For 4 matrices there are 5 possible parenthesizations, which we compare in the next section to check the answer.

In [4]:
print(f"Mincost: {Min_multi(matrixes)}")

------------------------------------------------------------------------------
[[0 0 1 1]
 [0 0 1 1]
 [0 0 0 2]
 [0 0 0 0]]
------------------------------------------------------------------------------
Mincost: 3600.0


## 5. Results and Analysis

### Output
```
3600.0
```
The minimum number of scalar multiplications needed to compute `A · B · C · D` is **3600**.

### Verification by brute force
With 4 matrices there are only 5 parenthesizations, so we can check the result by hand using `p = [10, 20, 9, 15, 8]`:

| Parenthesization | Calculation | Total cost |
|---|---|---|
| `((A·B)·C)·D` | 10·20·9 + 10·9·15 + 10·15·8 = 1800 + 1350 + 1200 | 4350 |
| `(A·(B·C))·D` | 20·9·15 + 10·20·15 + 10·15·8 = 2700 + 3000 + 1200 | 6900 |
| `A·((B·C)·D)` | 20·9·15 + 20·15·8 + 10·20·8 = 2700 + 2400 + 1600 | 6700 |
| `A·(B·(C·D))` | 9·15·8 + 20·9·8 + 10·20·8 = 1080 + 1440 + 1600 | 4120 |
| **`(A·B)·(C·D)`** | 10·20·9 + 9·15·8 + 10·9·8 = 1800 + 1080 + 720 | **3600** |

The best parenthesization is **`(A·B)·(C·D)`** with a cost of **3600**, which matches the program's output.

### Why it matters
The worst order costs **6900** multiplications and the best costs **3600**, so a poor choice does almost twice the work. For longer chains with larger matrices, the gap can become enormous, and trying every parenthesization is infeasible because their number grows exponentially (Catalan numbers). The DP approach finds the optimum in O(n³) time.

### Complexity
- **Time:** O(n³), from three nested loops.
- **Space:** O(n²), for the DP table.

### Conclusion
We used dynamic programming to compute the cheapest way to multiply a chain of matrices. For the matrices `A`, `B`, `C`, `D` the minimum cost is **3600 scalar multiplications**, achieved by grouping the chain as `(A·B)·(C·D)`.

